# 04. Валидация против Obu 2019 / ESA CCI / 33 бурений

**Цель:** оценить качество модели на независимых источниках. Это финальный отчётный
ноутбук, метрики из которого идут в Главу 5.

**Входы:**
- `results/maps/predictions_2014_2024.npz` (из 03)
- `data/y_new_rk_landcover.npz` — для per-zone (target, не валидация)
- `data/obu_2019_reprojected.npz` — карта Obu в нашей сетке
- `data/esa_cci_2023.npz` или подобный — ESA CCI (опционально)
- `data/boreholes/clean_boreholes_33.csv` — 33 чистых бурения

**Выходы:**
- `results/metrics/validation_summary.csv` — таблица для отчёта
- `results/metrics/per_zone_metrics.csv` — по зонам мерзлоты
- `results/figures/validation_panels.png` — 6-панельный слайд для защиты

In [ ]:
# Environment auto-detection
import os
from pathlib import Path

IN_COLAB_VM = (
    'COLAB_RELEASE_TAG' in os.environ or
    'COLAB_GPU' in os.environ
)

if IN_COLAB_VM:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = Path('/content/drive/MyDrive/AI4Arctic')
    env_label = 'Colab VM'
else:
    BASE_DIR = Path(os.environ.get(
        'AI4ARCTIC_HOME',
        Path.home() / 'Ai4Arctic'
    ))
    env_label = 'Local runtime'

print(f"Environment: {env_label}")
print(f"BASE_DIR: {BASE_DIR}")
assert BASE_DIR.exists(), f"BASE_DIR не найден: {BASE_DIR}"

import sys
sys.path.insert(0, str(BASE_DIR))

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f"Device: {device}")

DATA_DIR = BASE_DIR / 'data'
MODELS_DIR = BASE_DIR / 'models'
RESULTS_DIR = BASE_DIR / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'
METRICS_DIR = RESULTS_DIR / 'metrics'
MAPS_DIR = RESULTS_DIR / 'maps'

for d in [MODELS_DIR, FIGURES_DIR, METRICS_DIR, MAPS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 1. Загружаем предсказания и эталоны

In [ ]:
import pandas as pd

# Предсказания
preds_npz = np.load(MAPS_DIR / 'predictions_2014_2024.npz')
pred_2023 = preds_npz['pred_2023']
pred_2023_lh = preds_npz['pred_2023_lh']
lats = preds_npz['lats']; lons = preds_npz['lons']

print(f"pred_2023 (raw):     mean {np.nanmean(pred_2023):+.3f}, range [{np.nanmin(pred_2023):+.2f}, {np.nanmax(pred_2023):+.2f}]")
print(f"pred_2023_lh (MAGT): mean {np.nanmean(pred_2023_lh):+.3f}, range [{np.nanmin(pred_2023_lh):+.2f}, {np.nanmax(pred_2023_lh):+.2f}]")

# Target из y_new (для per-zone — это НЕ валидация, это сам target обучения)
y_new = np.load(DATA_DIR / 'y_new_rk_landcover.npz')['y_new']
real_2023 = y_new[13]  # индекс 13 = 2023
print(f"\ntarget y_new[13] (2023): mean {np.nanmean(real_2023):+.3f}")

## 2. Per-zone метрики (ИСПРАВЛЕННАЯ версия)

`pred` (без LH) vs `real` (raw target из y_new). Зоны — по `pred_lh`.

In [ ]:
from src.metrics import per_zone_metrics, per_zone_metrics_to_csv

zone_results = per_zone_metrics(
    pred=pred_2023,
    real=real_2023,
    pred_lh=pred_2023_lh,
    verbose=True,
)

per_zone_metrics_to_csv(zone_results, METRICS_DIR / 'per_zone_metrics.csv')
print(f"\nСохранено: {METRICS_DIR / 'per_zone_metrics.csv'}")

## 3. Сравнение с Obu 2019

In [ ]:
from src.metrics import all_metrics

OBU_PATH = DATA_DIR / 'obu_2019_reprojected.npz'

if OBU_PATH.exists():
    obu = np.load(OBU_PATH)['obu_on_grid']
    print(f"Obu 2019: mean {np.nanmean(obu):+.3f}")
    obu_metrics = all_metrics(pred_2023_lh, obu, label='vs Obu 2019', return_dict=True)
    print()
    for k, v in obu_metrics.items():
        print(f"  {k}: {v}")
else:
    print(f"Obu не найден в {OBU_PATH}")
    obu_metrics = None

## 4. Сравнение с ESA CCI (если доступно)

In [ ]:
ESA_PATH = DATA_DIR / 'comparison_esacci_2023.npz'

if ESA_PATH.exists():
    esa = np.load(ESA_PATH)['esa_on_grid']
    print(f"ESA CCI: mean {np.nanmean(esa):+.3f}")
    esa_metrics = all_metrics(pred_2023_lh, esa, label='vs ESA CCI', return_dict=True)
    print()
    for k, v in esa_metrics.items():
        print(f"  {k}: {v}")
else:
    print(f"ESA не найден в {ESA_PATH}")
    esa_metrics = None

## 5. Сравнение с 33 in-situ бурениями

In [ ]:
BOREHOLES_PATH = DATA_DIR / 'boreholes' / 'clean_boreholes_33.csv'

if BOREHOLES_PATH.exists():
    boreholes = pd.read_csv(BOREHOLES_PATH)
    print(f"Бурений: {len(boreholes)}")
    
    def find_pixel(lat, lon):
        return np.argmin(np.abs(lats - lat)), np.argmin(np.abs(lons - lon))
    
    boreholes['pred_lh'] = [pred_2023_lh[find_pixel(r['lat'], r['lon'])]
                             for _, r in boreholes.iterrows()]
    
    # ВНИМАНИЕ: имя колонки с observed MAGT — проверь у себя
    obs_col = 'magt_adjusted' if 'magt_adjusted' in boreholes.columns else 'magt_obs'
    
    borehole_metrics = all_metrics(
        boreholes['pred_lh'].values, boreholes[obs_col].values,
        label=f'vs {len(boreholes)} boreholes', return_dict=True
    )
    print()
    for k, v in borehole_metrics.items():
        print(f"  {k}: {v}")
else:
    print(f"Бурения не найдены в {BOREHOLES_PATH}")
    boreholes = None
    borehole_metrics = None

## 6. Сводная таблица метрик

In [ ]:
summary_rows = []
for label, metrics in [('Obu 2019', obu_metrics),
                       ('ESA CCI', esa_metrics),
                       ('33 boreholes', borehole_metrics)]:
    if metrics is not None:
        row = {'reference': label, **metrics}
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(METRICS_DIR / 'validation_summary.csv', index=False)
print(summary_df.to_string(index=False))

## 7. 6-панельная визуализация (слайд защиты)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 11))

# 1: MAGT predicted
im0 = axes[0, 0].pcolormesh(lons, lats, pred_2023_lh, cmap='RdBu_r', vmin=-12, vmax=2)
axes[0, 0].set_title('MAGT predicted, 2023')
plt.colorbar(im0, ax=axes[0, 0], fraction=0.04, label='°C')

# 2: Obu (если есть)
if obu_metrics:
    obu = np.load(OBU_PATH)['obu_on_grid']
    im1 = axes[0, 1].pcolormesh(lons, lats, obu, cmap='RdBu_r', vmin=-12, vmax=2)
    axes[0, 1].set_title('Obu 2019 reference')
    plt.colorbar(im1, ax=axes[0, 1], fraction=0.04, label='°C')

# 3: difference
if obu_metrics:
    diff = pred_2023_lh - obu
    im2 = axes[0, 2].pcolormesh(lons, lats, diff, cmap='seismic', vmin=-4, vmax=4)
    axes[0, 2].set_title(f'pred - Obu (bias={obu_metrics["Bias"]:+.2f})')
    plt.colorbar(im2, ax=axes[0, 2], fraction=0.04, label='°C')

# 4: scatter vs Obu
if obu_metrics:
    mask = ~np.isnan(pred_2023_lh) & ~np.isnan(obu)
    axes[1, 0].scatter(obu[mask][::100], pred_2023_lh[mask][::100], s=1, alpha=0.3)
    lim = [-15, 5]
    axes[1, 0].plot(lim, lim, 'r--', alpha=0.7)
    axes[1, 0].set_xlim(lim); axes[1, 0].set_ylim(lim)
    axes[1, 0].set_xlabel('Obu 2019'); axes[1, 0].set_ylabel('Predicted')
    axes[1, 0].set_title(f'vs Obu: RMSE={obu_metrics["RMSE"]:.2f}, r={obu_metrics["r"]:+.3f}')
    axes[1, 0].grid(alpha=0.3)

# 5: scatter vs boreholes
if boreholes is not None and borehole_metrics:
    obs_col = 'magt_adjusted' if 'magt_adjusted' in boreholes.columns else 'magt_obs'
    axes[1, 1].scatter(boreholes[obs_col], boreholes['pred_lh'], s=50,
                        edgecolor='k', alpha=0.7)
    lim = [-15, 5]
    axes[1, 1].plot(lim, lim, 'r--', alpha=0.7)
    axes[1, 1].set_xlim(lim); axes[1, 1].set_ylim(lim)
    axes[1, 1].set_xlabel('Borehole MAGT (observed)'); axes[1, 1].set_ylabel('Predicted')
    axes[1, 1].set_title(f'vs {len(boreholes)} boreholes: RMSE={borehole_metrics["RMSE"]:.2f}')
    axes[1, 1].grid(alpha=0.3)

# 6: per-zone bar chart
if zone_results:
    df_zones = pd.DataFrame(zone_results)
    axes[1, 2].barh(df_zones['Зона'], df_zones['RMSE °C'], color='steelblue')
    axes[1, 2].set_xlabel('RMSE, °C')
    axes[1, 2].set_title('Per-zone RMSE (pred vs target)')
    for i, v in enumerate(df_zones['RMSE °C']):
        axes[1, 2].text(v + 0.02, i, f'{v:.2f}', va='center')

plt.suptitle('Финальная модель: ConvLSTM v2 с rk-landcover + latent heat', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'validation_panels.png', dpi=150, bbox_inches='tight')
plt.show()